In [ ]:
import numpy as np
import cv2
import os
import imutils
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.inception_v3 import InceptionV3
from sklearn.metrics import classification_report, confusion_matrix
import itertools
import time
import pandas as pd
from tensorflow.keras.models import load_model

In [ ]:
def crop_img(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (3, 3), 0)

    bimg = cv2.threshold(gray, 45, 255, cv2.THRESH_BINARY)[1]
    bimg = cv2.erode(bimg, None, iterations=2)
    bimg = cv2.dilate(bimg, None, iterations=2)

    countour_list = cv2.findContours(bimg.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    countour_list = imutils.grab_contours(countour_list)

    countour = max(countour_list, key=cv2.contourArea)

    left_point = tuple(countour[countour[:, :, 0].argmin()][0])
    right_point = tuple(countour[countour[:, :, 0].argmax()][0])
    top_point = tuple(countour[countour[:, :, 1].argmin()][0])
    bottom_point = tuple(countour[countour[:, :, 1].argmax()][0])

    ADD_PIXELS = 0
    start_y = max(0, top_point[1] - ADD_PIXELS)
    end_y = min(img.shape[0], bottom_point[1] + ADD_PIXELS)
    start_x = max(0, left_point[0] - ADD_PIXELS)
    end_x = min(img.shape[1], right_point[0] + ADD_PIXELS)

    new_img = img[start_y:end_y, start_x:end_x].copy()

    return new_img

def VizGradCAM(model, image, plot_results=True, save_path=None, dpi=300):
    img_tensor = np.expand_dims(image, axis=0)

    last_conv_layer_name = 'mixed10'
    last_conv_layer = model.get_layer(last_conv_layer_name)

    grad_model = tf.keras.models.Model(
        [model.inputs], [last_conv_layer.output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, predictions = grad_model(img_tensor)
        predicted_class_idx = tf.argmax(predictions[0])
        class_channel = predictions[:, predicted_class_idx]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]

    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / tf.reduce_max(heatmap)
    heatmap_np = heatmap.numpy()

    heatmap_visual = cv2.resize(heatmap_np, (image.shape[1], image.shape[0]))
    heatmap_visual = np.uint8(255 * heatmap_visual)
    heatmap_color = cv2.applyColorMap(heatmap_visual, cv2.COLORMAP_JET)

    display_image = (image * 255).astype(np.uint8)
    if display_image.shape[-1] == 1:
        display_image = cv2.cvtColor(display_image, cv2.COLOR_GRAY2BGR)

    overlaid_image = cv2.addWeighted(display_image, 0.6, heatmap_color, 0.4, 0)

    if plot_results or save_path:
        fig, axes = plt.subplots(1, 3, figsize=(12, 4))

        original_rgb = cv2.cvtColor(display_image, cv2.COLOR_BGR2RGB)
        heatmap_rgb = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB)
        overlaid_rgb = cv2.cvtColor(overlaid_image, cv2.COLOR_BGR2RGB)

        axes[0].set_title('Original Image')
        axes[0].imshow(original_rgb)
        axes[0].axis('off')

        axes[1].set_title('Grad-CAM Heatmap')
        axes[1].imshow(heatmap_rgb)
        axes[1].axis('off')

        axes[2].set_title('Overlaid Image')
        axes[2].imshow(overlaid_rgb)
        axes[2].axis('off')

        if save_path:
            output_dir = os.path.dirname(save_path)
            if output_dir and not os.path.exists(output_dir):
                os.makedirs(output_dir)
            plt.tight_layout()
            plt.savefig(save_path, dpi=dpi, bbox_inches='tight', pad_inches=0.1)

        if plot_results:
            plt.show()

        plt.close(fig)

    return overlaid_image, heatmap_np

def plot_confusion_matrix(cm, classes, normalize=False, title='Confusion matrix', cmap=plt.cm.Blues):
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i,
                 f'{cm[i, j]:.2f}' if normalize else f'{cm[i, j]}',
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
training_dir_path = "Dataset/Training"
testing_dir_path = "Dataset/Testing"
IMG_SIZE = 224
labels = ['glioma', 'meningioma', 'notumor', 'pituitary']

cropped_training_path = 'Dataset/cropped/Training'
cropped_testing_path = 'Dataset/cropped/Testing'

for label_dir in labels:
    os.makedirs(os.path.join(cropped_training_path, label_dir), exist_ok=True)
    os.makedirs(os.path.join(cropped_testing_path, label_dir), exist_ok=True)

print("Processing and cropping training images:")
for label in labels:
    folderpath = os.path.join(training_dir_path, label)
    image_files = os.listdir(folderpath)
    for img_file in image_files:
        image_path = os.path.join(folderpath, img_file)
        image = cv2.imread(image_path)
        new_img = crop_img(image)
        new_img = cv2.resize(new_img, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(os.path.join(cropped_training_path, label, img_file), new_img)

print("Processing and cropping testing images:")
for label in labels:
    folderpath = os.path.join(testing_dir_path, label)
    image_files = os.listdir(folderpath)
    for img_file in image_files:
        image_path = os.path.join(folderpath, img_file)
        image = cv2.imread(image_path)
        new_img = crop_img(image)
        new_img = cv2.resize(new_img, (IMG_SIZE, IMG_SIZE))
        cv2.imwrite(os.path.join(cropped_testing_path, label, img_file), new_img)


In [ ]:
X_train = []
y_train = []
X_testval = []
y_testval = []

print("Loading cropped training data:")
for label_idx, label in enumerate(labels):
    folderpath = os.path.join(cropped_training_path, label)
    for img_file in os.listdir(folderpath):
        img_path = os.path.join(folderpath, img_file)
        image = cv2.imread(img_path, 0)
        image = cv2.bilateralFilter(image, 2, 50, 50)
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE)
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        X_train.append(image)
        y_train.append(label_idx)

print("Loading cropped testing data:")
for label_idx, label in enumerate(labels):
    folderpath = os.path.join(cropped_testing_path, label)
    for img_file in os.listdir(folderpath):
        img_path = os.path.join(folderpath, img_file)
        image = cv2.imread(img_path, 0)
        image = cv2.bilateralFilter(image, 2, 50, 50)
        image = cv2.applyColorMap(image, cv2.COLORMAP_BONE)
        image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))
        X_testval.append(image)
        y_testval.append(label_idx)

X_train = np.array(X_train, dtype=np.float32) / 255.0
y_train = np.array(y_train, dtype=np.int32)
X_testval = np.array(X_testval, dtype=np.float32) / 255.0
y_testval = np.array(y_testval, dtype=np.int32)

X_val, X_test, y_val, y_test = train_test_split(X_testval, y_testval, test_size=0.5, random_state=42, stratify=y_testval)


num_classes = len(labels)
y_train_one_hot = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
y_val_one_hot = tf.keras.utils.to_categorical(y_val, num_classes=num_classes)
y_test_one_hot = tf.keras.utils.to_categorical(y_test, num_classes=num_classes)

print("\nData shapes after split:")
print("Train X:", X_train.shape, "y:", y_train_one_hot.shape)
print("Validation X:", X_val.shape, "y:", y_val_one_hot.shape)
print("Test X:", X_test.shape, "y:", y_test_one_hot.shape)

print("\nPer-class counts:")
for idx, label in enumerate(labels):
    train_count = int(np.sum(y_train == idx))
    val_count = int(np.sum(y_val == idx))
    test_count = int(np.sum(y_test == idx))
    print(f"  {label:12s} -> train: {train_count:3d}, val: {val_count:3d}, test: {test_count:3d}")

In [ ]:
print("\nDataset Distribution Table:")
data_distribution = {
    'Dataset': labels,
    'Training': [0] * len(labels),
    'Validation': [0] * len(labels),
    'Testing': [0] * len(labels)
}

for i, label_name in enumerate(labels):
    data_distribution['Training'][i] = np.sum(y_train == i)
    data_distribution['Validation'][i] = np.sum(y_val == i)
    data_distribution['Testing'][i] = np.sum(y_test == i)

df_distribution = pd.DataFrame(data_distribution)

df_distribution.loc['Total'] = df_distribution.sum(numeric_only=True)
df_distribution.loc['Total', 'Dataset'] = 'Total'

df_distribution = df_distribution.set_index('Dataset')

print(df_distribution.to_string())

plt.figure(figsize=(8, 6))
plt.text(0.01, 0.99, df_distribution.to_string(), {'fontsize': 12}, fontproperties='monospace', va='top')
plt.axis('off')
plt.title("Dataset Distribution", fontsize=16)
plt.savefig('InceptionAug25_Dataset_Distribution.png', format='png', dpi=300, bbox_inches='tight')
plt.close()
df_distribution.to_csv('InceptionAug25_Dataset_Distribution.csv')

In [ ]:
data_aug = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.05,
    height_shift_range=0.05,
    horizontal_flip=True
)

data_aug.fit(X_train)

In [ ]:
num_samples_to_show = 4

fig, axes = plt.subplots(2, num_samples_to_show, figsize=(15, 8))

plt.subplots_adjust(hspace=0.3, wspace=0.05, top=0.88)

original_images_for_viz = X_train[:num_samples_to_show]

augmented_images_for_viz = []
for i in range(num_samples_to_show):
    original_img_batch = np.expand_dims(original_images_for_viz[i], axis=0)

    augmented_batch = iter(data_aug.flow(original_img_batch, np.array([0]), batch_size=1)).__next__()
    augmented_img = augmented_batch[0][0]
    augmented_images_for_viz.append(augmented_img)


for i in range(num_samples_to_show):
    axes[0, i].imshow(original_images_for_viz[i])
    axes[0, i].set_title(f"({chr(97+i)}) Original", fontsize=12, pad=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(augmented_images_for_viz[i])
    axes[1, i].set_title(f"({chr(97+i)}')$'$ Augmented", fontsize=12, pad=10)
    axes[1, i].axis('off')

plt.suptitle("Brain Tumor MRI Before and After Augmentation", fontsize=16, y=0.95)

plt.savefig('InceptionAug_Augmentation_Examples.png', format='png', dpi=300)
plt.show()

In [ ]:
IMG_SIZE_MODEL = (224, 224)

base_model = InceptionV3(
    include_top=False,
    input_shape=IMG_SIZE_MODEL + (3,),
    weights='imagenet'
)

for layer in base_model.layers:
    layer.trainable = False

for layer in base_model.layers[-40:]:
    layer.trainable = True

base_model.summary()

In [ ]:
x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.4)(x)
x = Dense(800, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(200, activation='relu')(x)
x = Dropout(0.4)(x)
x = Dense(50, activation='relu')(x)

predict = Dense(len(labels), activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predict)

model.summary()

adam = Adam(learning_rate=0.0001)
model.compile(optimizer=adam, loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
hist = model.fit(
    data_aug.flow(X_train, y_train_one_hot, batch_size=32),
    validation_data=(X_val, y_val_one_hot),
    epochs=25,
    verbose=1
)

In [ ]:
model.save('InceptionAug25Grad.h5')

In [ ]:
acc = hist.history['accuracy']
val_acc = hist.history['val_accuracy']
loss = hist.history['loss']
val_loss = hist.history['val_loss']

plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc="lower right")
plt.title("Training and Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(True, linestyle='--', alpha=0.6)


plt.subplot(1, 2, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc="upper right")
plt.title("Training and Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('InceptionAug25AccLoss.png', format='png', dpi=300)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelBinarizer

model = load_model('InceptionAug25Grad.h5')
y_pred_probs = model.predict(X_test, verbose=1)
plt.figure(figsize=(10, 8))

fpr = dict()
tpr = dict()
roc_auc = dict()

n_classes = len(labels)

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_one_hot[:, i], y_pred_probs[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

    plt.plot(fpr[i], tpr[i], label=f'ROC curve of {labels[i]} (area = {roc_auc[i]:.2f})')

plt.plot([0, 1], [0, 1], 'k--', label='Chance level (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves for Each Class')
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig('InceptionAugROC_Curve.png', format='png', dpi=300)
plt.show()

In [ ]:
model = load_model('InceptionAug25Grad.h5')
loss, acc = model.evaluate(X_test, y_test_one_hot, verbose=1)
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {acc:.4f}")

print("\nGenerating classification report for training data...")
predicted_train_classes = np.argmax(model.predict(X_train, verbose=0), axis=1)
true_train_classes = np.argmax(y_train_one_hot, axis=1)
report_train = classification_report(true_train_classes, predicted_train_classes, target_names=labels, output_dict=True)
print(classification_report(true_train_classes, predicted_train_classes, target_names=labels))
df_report_train = pd.DataFrame(report_train).transpose()
df_report_train.to_csv('InceptionAug25_classification_report_train.csv', index=True)

print("\nGenerating classification report for test data...")
predicted_test_classes = np.argmax(model.predict(X_test, verbose=0), axis=1)
true_test_classes = np.argmax(y_test_one_hot, axis=1)
report_test = classification_report(true_test_classes, predicted_test_classes, target_names=labels, output_dict=True)
print(classification_report(true_test_classes, predicted_test_classes, target_names=labels))
df_report_test = pd.DataFrame(report_test).transpose()
df_report_test.to_csv('InceptionAug25classification_report_test.csv', index=True)

In [ ]:
performance_data = []
for label_name in labels:
    if label_name in report_test:
        precision = report_test[label_name]['precision']
        recall = report_test[label_name]['recall']
        f1_score = report_test[label_name]['f1-score']
        performance_data.append({
            'Models': 'InceptionV3',
            'Class': label_name.capitalize(),
            'Precision': f"{precision:.2f}",
            'Recall': f"{recall:.2f}",
            'F1-Score': f"{f1_score:.2f}"
        })

overall_accuracy = report_test['accuracy']
weighted_precision = report_test['weighted avg']['precision']
weighted_recall = report_test['weighted avg']['recall']
weighted_f1_score = report_test['weighted avg']['f1-score']

if performance_data:
    performance_data[0]['Accuracy'] = f"{overall_accuracy*100:.2f}%"
    for i in range(1, len(performance_data)):
        performance_data[i]['Accuracy'] = ""

performance_data.append({
    'Models': '',
    'Class': 'Weighted Avg',
    'Precision': f"{weighted_precision:.2f}",
    'Recall': f"{weighted_recall:.2f}",
    'F1-Score': f"{weighted_f1_score:.2f}",
    'Accuracy': ""
})

df_performance = pd.DataFrame(performance_data)
df_performance = df_performance[['Models', 'Class', 'Precision', 'Recall', 'F1-Score', 'Accuracy']]

print(df_performance.to_string(index=False))

df_performance.to_csv('InceptionAug25_Model_Performance.csv', index=False)

In [ ]:
print("\nGenerating confusion matrix...")
confusion_mtx = confusion_matrix(true_test_classes, predicted_test_classes)

plt.figure(figsize=(8, 6))
plot_confusion_matrix(confusion_mtx, classes=labels, title='Confusion Matrix')
plt.savefig('InceptionAug25CM.png', format='png', dpi=300)
plt.show()

In [ ]:

no_tumor_true_idx = labels.index('notumor')
glioma_pred_idx = labels.index('glioma')

if no_tumor_true_idx != -1 and glioma_pred_idx != -1:
    misclassified_indices = []
    for i in range(len(true_test_classes)):
        if (true_test_classes[i] == no_tumor_true_idx and
            predicted_test_classes[i] == glioma_pred_idx):
            misclassified_indices.append(i)

    if misclassified_indices:
        num_misclassified = len(misclassified_indices)
        print(f"Found {num_misclassified} instances of no-tumor misdiagnosed as glioma. Displaying up to 8 of them.")

        display_count = min(num_misclassified, 8)
        indices_to_display = np.random.choice(misclassified_indices, size=display_count, replace=False)

        fig, axes = plt.subplots(2, 4, figsize=(15, 8))
        axes = axes.flatten()

        for i, idx in enumerate(indices_to_display):
            ax = axes[i]
            img = X_test[idx]
            ax.imshow(img)
            ax.set_title(f"True: {labels[true_test_classes[idx]]}\nPred: {labels[predicted_test_classes[idx]]}",
                         color='orange', fontsize=10)
            ax.axis('off')

        for j in range(display_count, 8):
            fig.delaxes(axes[j])

        plt.suptitle("No-tumor Images Misdiagnosed as Glioma", fontsize=16, y=1.02)
        plt.tight_layout()
        plt.savefig('InceptionAug_Misdiagnosed_NoTumor_Glioma.png', format='png', dpi=300)
        plt.show()
    else:
        print("No instances of no-tumor misdiagnosed as glioma found.")
else:
    print("Misdiagnosis plot skipped due to label configuration.")

In [ ]:
y_hat = model.predict(X_test, verbose=0)

fig = plt.figure(figsize=(20, 8))
num_samples_to_show = min(12, X_test.shape[0])
random_indices = np.random.choice(X_test.shape[0], size=num_samples_to_show, replace=False)

for i, idx in enumerate(random_indices):
    ax = fig.add_subplot(4,4, i+1, xticks=[], yticks=[])
    ax.imshow(X_test[idx])
    pred_idx = np.argmax(y_hat[idx])
    true_idx = np.argmax(y_test_one_hot[idx])
    ax.set_title("{} ({})".format(labels[pred_idx], labels[true_idx]),
                 color=("blue" if pred_idx == true_idx else "orange"))
    ax.axis('off')

plt.suptitle("Sample Predictions (True vs. Predicted)", fontsize=16)
plt.tight_layout()
plt.savefig('InceptionAug_Sample_Predictions.png', format='png', dpi=300)
plt.show()

In [ ]:
inference_times_ms = []

for i in range(X_test.shape[0]):
    img = X_test[i]
    img_batch = np.expand_dims(img, axis=0)

    start_time = time.time()
    _ = model.predict(img_batch, verbose=0)
    end_time = time.time()

    duration_ms = (end_time - start_time) * 1000
    inference_times_ms.append(duration_ms)

min_time_ms = np.min(inference_times_ms)
max_time_ms = np.max(inference_times_ms)
avg_time_ms = np.mean(inference_times_ms)

inference_results = pd.DataFrame({
    'Model': ['InceptionV3'],
    'Min. Time (ms)': [f"{min_time_ms:.1f}"],
    'Max. Time (ms)': [f"{max_time_ms:.1f}"],
    'Avg. Time (ms)': [f"{avg_time_ms:.1f}"]
})

print(inference_results.to_string(index=False))
inference_results.to_csv('InceptionAug25_Model_Inference_Time.csv', index=False)

In [ ]:
from sklearn.metrics import pairwise_distances
from skimage.metrics import structural_similarity as ssim

def compute_fidelity(model, image, heatmap, pred_idx, top_percent=0.2):
    heatmap_resized = cv2.resize(heatmap, (image.shape[1], image.shape[0]))

    flat_heatmap = heatmap_resized.flatten()
    threshold = np.percentile(flat_heatmap, 100 * (1 - top_percent))
    mask = (heatmap_resized >= threshold).astype(np.float32)
    mask = np.repeat(mask[:, :, np.newaxis], 3, axis=-1)

    masked_image = image * mask

    original_prob = model.predict(np.expand_dims(image, axis=0), verbose=0)[0][pred_idx]
    masked_prob = model.predict(np.expand_dims(masked_image, axis=0), verbose=0)[0][pred_idx]

    return original_prob - masked_prob

def compute_stability(model, image, original_heatmap, epsilon=0.05):
    noisy_image = image + np.random.normal(0, epsilon, image.shape)
    noisy_image = np.clip(noisy_image, 0, 1)

    _, noisy_heatmap = VizGradCAM(model, noisy_image, plot_results=False)

    orig_gray = (original_heatmap * 255).astype(np.uint8)
    noisy_gray = (noisy_heatmap * 255).astype(np.uint8)

    min_dim = min(orig_gray.shape[0], orig_gray.shape[1])
    win_size = min(7, min_dim if min_dim % 2 == 1 else min_dim - 1)

    score, _ = ssim(orig_gray, noisy_gray, win_size=win_size, channel_axis=None, full=True)
    return score

def compute_consistency(heatmaps):
    flat_hm = [hm.flatten() for hm in heatmaps]
    consistency_scores = 1 - pairwise_distances(flat_hm, metric='cosine')
    return np.mean(consistency_scores)

In [ ]:
model = load_model('InceptionAug25Grad.h5')
GRADCAM_OUTPUT_DIR = 'gradcam_visualizations'
if not os.path.exists(GRADCAM_OUTPUT_DIR):
    os.makedirs(GRADCAM_OUTPUT_DIR)

num_gradcam_viz = 90

metrics_list = []
heatmaps = []
model_preds = []


random_gradcam_indices = np.random.choice(X_test.shape[0], size=num_gradcam_viz, replace=False)

for i, idx in enumerate(random_gradcam_indices):
    image_to_viz = X_test[idx]
    true_idx = np.argmax(y_test_one_hot[idx])
    true_label = labels[true_idx]

    pred_probs = model.predict(np.expand_dims(image_to_viz, axis=0), verbose=0)
    pred_idx = np.argmax(pred_probs)
    pred_label = labels[pred_idx]
    confidence = pred_probs[0][pred_idx]

    print(f"\nImage #{idx} ({i+1}/{num_gradcam_viz}) | True: {true_label} | Pred: {pred_label} ({confidence:.2f})")

    save_filename = os.path.join(
        GRADCAM_OUTPUT_DIR,
        f"gradcam_image_{idx}_true_{true_label.replace(' ', '_')}_pred_{pred_label.replace(' ', '_')}.jpeg"
    )

    _, heatmap = VizGradCAM(model, image_to_viz, plot_results=True, save_path=save_filename, dpi=300)

    fidelity = compute_fidelity(model, image_to_viz, heatmap, pred_idx)
    stability = compute_stability(model, image_to_viz, heatmap)

    heatmaps.append(heatmap)
    model_preds.append(pred_idx)

    metrics_list.append({
        "Index": idx,
        "True Label": true_label,
        "Predicted Label": pred_label,
        "Confidence": confidence,
        "Fidelity": fidelity,
        "Stability": stability
    })

    print(f"Saved Grad-CAM to {save_filename}")

consistency_score = compute_consistency(heatmaps)
print(f"\nConsistency Score: {consistency_score:.4f}")

import pandas as pd
results_df = pd.DataFrame(metrics_list)
results_df.to_csv("gradcam_per_image_scores.csv", index=False)

avg_fidelity = np.mean([m["Fidelity"] for m in metrics_list])
avg_stability = np.mean([m["Stability"] for m in metrics_list])
summary_df = pd.DataFrame({
    'Metric': ['Fidelity', 'Stability', 'Consistency'],
    'Score': [avg_fidelity, avg_stability, consistency_score]
})
summary_df.to_csv("gradcam_summary_scores.csv", index=False)


In [ ]:
model = load_model('InceptionAug25Grad.h5')

TEST_DIR = "test"
GRADCAM_OUTPUT_DIR = "gradcam_visualizations"
labels = ["glioma", "meningioma", "no_tumor", "pituitary"]

for label in labels:
    os.makedirs(os.path.join(GRADCAM_OUTPUT_DIR, label), exist_ok=True)

IMG_SIZE = 224

def preprocess_image(image_path):
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = crop_img(img)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0
    return img

test_images = []
test_labels = []
image_paths = []

for label in labels:
    folder = os.path.join(TEST_DIR, label)
    for fname in os.listdir(folder):
        fpath = os.path.join(folder, fname)
        if fname.lower().endswith((".png", ".jpg", ".jpeg")):
            processed_img = preprocess_image(fpath)
            if processed_img is not None:
                test_images.append(processed_img)
                test_labels.append(label)
                image_paths.append(fpath)

test_images = np.array(test_images)

metrics_list = []
heatmaps = []
model_preds = []

for i, (image_to_viz, true_label, path) in enumerate(zip(test_images, test_labels, image_paths)):
    pred_probs = model.predict(np.expand_dims(image_to_viz, axis=0), verbose=0)
    pred_idx = np.argmax(pred_probs)
    pred_label = labels[pred_idx]
    confidence = pred_probs[0][pred_idx]

    print(f"\nImage {i+1}/{len(test_images)} | True: {true_label} | Pred: {pred_label} ({confidence:.2f})")

    save_filename = os.path.join(
        GRADCAM_OUTPUT_DIR,
        true_label,
        f"{os.path.basename(path).split('.')[0]}_true_{true_label}_pred_{pred_label}.png"
    )

    _, heatmap = VizGradCAM(model, image_to_viz, plot_results=True, save_path=save_filename, dpi=300)

    fidelity = compute_fidelity(model, image_to_viz, heatmap, pred_idx)
    stability = compute_stability(model, image_to_viz, heatmap)

    heatmaps.append(heatmap)
    model_preds.append(pred_idx)

    metrics_list.append({
        "Image": os.path.basename(path),
        "True Label": true_label,
        "Predicted Label": pred_label,
        "Confidence": confidence,
        "Fidelity": fidelity,
        "Stability": stability
    })


consistency_score = compute_consistency(heatmaps)
print(f"\nConsistency Score: {consistency_score:.4f}")

results_df = pd.DataFrame(metrics_list)
results_df.to_csv("gradcam_per_image_scores.csv", index=False)

avg_fidelity = np.mean([m["Fidelity"] for m in metrics_list])
avg_stability = np.mean([m["Stability"] for m in metrics_list])
summary_df = pd.DataFrame({
    'Metric': ['Fidelity', 'Stability', 'Consistency'],
    'Score': [avg_fidelity, avg_stability, consistency_score]
})
summary_df.to_csv("gradcam_summary_scores.csv", index=False)